# LGBM

### 서울

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1)
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_서울특별시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (LightGBM)
print(f"{'='*30}\nSTART: LIGHTGBM (SAMPLE-WISE SPLIT)\n{'='*30}")

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,           # XGBoost와 동일하게 0.05
    num_leaves=31,                # LGBM의 핵심 파라미터 (max_depth와 대응)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    importance_type='gain',
    verbosity=-1                  # 불필요한 경고 메시지 출력 방지
)

# 6. 학습 (Early Stopping 설정 방식이 다름)
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20), # 여기서 조기종료 설정
        lgb.log_evaluation(period=0)            # verbose=False와 동일 효과
    ]
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LIGHTGBM (SAMPLE-WISE SPLIT)
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[361]	valid_0's rmse: 0.19868	valid_0's l2: 0.0394739
Validation 결과 | R2: 0.9826 | MAE: 4,342 | RMSE: 7,508 | MAPE: 4.21% | MdAPE: 3.16% | RMSLE: 0.0589
------------------------------
FINAL TEST RESULT: R2: 0.9237 | MAE: 10,420 | RMSE: 17,795 | MAPE: 8.78% | MdAPE: 5.93% | RMSLE: 0.1187


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1)
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_서울특별시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (LightGBM)
print(f"{'='*30}\nSTART: LIGHTGBM (SAMPLE-WISE SPLIT)\n{'='*30}")

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,           # XGBoost와 동일하게 0.05
    num_leaves=31,                # LGBM의 핵심 파라미터 (max_depth와 대응)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    importance_type='gain',
    verbosity=-1                  # 불필요한 경고 메시지 출력 방지
)

# 6. 학습 (Early Stopping 설정 방식이 다름)
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20), # 여기서 조기종료 설정
        lgb.log_evaluation(period=0)            # verbose=False와 동일 효과
    ]
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LIGHTGBM (SAMPLE-WISE SPLIT)
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[301]	valid_0's rmse: 0.181528	valid_0's l2: 0.0329524
Validation 결과 | R2: 0.9847 | MAE: 3,596 | RMSE: 8,001 | MAPE: 2.71% | MdAPE: 1.81% | RMSLE: 0.0439
------------------------------
FINAL TEST RESULT: R2: 0.9521 | MAE: 8,324 | RMSE: 14,599 | MAPE: 6.56% | MdAPE: 5.54% | RMSLE: 0.0781


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1)
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_서울특별시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (LightGBM)
print(f"{'='*30}\nSTART: LIGHTGBM (SAMPLE-WISE SPLIT)\n{'='*30}")

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,           # XGBoost와 동일하게 0.05
    num_leaves=31,                # LGBM의 핵심 파라미터 (max_depth와 대응)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    importance_type='gain',
    verbosity=-1                  # 불필요한 경고 메시지 출력 방지
)

# 6. 학습 (Early Stopping 설정 방식이 다름)
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20), # 여기서 조기종료 설정
        lgb.log_evaluation(period=0)            # verbose=False와 동일 효과
    ]
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LIGHTGBM (SAMPLE-WISE SPLIT)
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[211]	valid_0's rmse: 0.195302	valid_0's l2: 0.0381428
Validation 결과 | R2: 0.9822 | MAE: 3,709 | RMSE: 8,606 | MAPE: 2.73% | MdAPE: 1.79% | RMSLE: 0.0448
------------------------------
FINAL TEST RESULT: R2: 0.9527 | MAE: 7,786 | RMSE: 14,486 | MAPE: 6.17% | MdAPE: 5.14% | RMSLE: 0.0756


### 부산

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1)
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_부산광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (LightGBM)
print(f"{'='*30}\nSTART: LIGHTGBM (SAMPLE-WISE SPLIT)\n{'='*30}")

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,           # XGBoost와 동일하게 0.05
    num_leaves=31,                # LGBM의 핵심 파라미터 (max_depth와 대응)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    importance_type='gain',
    verbosity=-1                  # 불필요한 경고 메시지 출력 방지
)

# 6. 학습 (Early Stopping 설정 방식이 다름)
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20), # 여기서 조기종료 설정
        lgb.log_evaluation(period=0)            # verbose=False와 동일 효과
    ]
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LIGHTGBM (SAMPLE-WISE SPLIT)
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[374]	valid_0's rmse: 0.205187	valid_0's l2: 0.0421016
Validation 결과 | R2: 0.9764 | MAE: 1,535 | RMSE: 3,102 | MAPE: 5.04% | MdAPE: 3.64% | RMSLE: 0.0707
------------------------------
FINAL TEST RESULT: R2: 0.8932 | MAE: 4,012 | RMSE: 8,742 | MAPE: 9.43% | MdAPE: 6.58% | RMSLE: 0.1386


In [ ]:
import os

import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1)
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_부산광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (LightGBM)
print(f"{'='*30}\nSTART: LIGHTGBM (SAMPLE-WISE SPLIT)\n{'='*30}")

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,           # XGBoost와 동일하게 0.05
    num_leaves=31,                # LGBM의 핵심 파라미터 (max_depth와 대응)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    importance_type='gain',
    verbosity=-1                  # 불필요한 경고 메시지 출력 방지
)

# 6. 학습 (Early Stopping 설정 방식이 다름)
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20), # 여기서 조기종료 설정
        lgb.log_evaluation(period=0)            # verbose=False와 동일 효과
    ]
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LIGHTGBM (SAMPLE-WISE SPLIT)
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[80]	valid_0's rmse: 0.293355	valid_0's l2: 0.0860572
Validation 결과 | R2: 0.9746 | MAE: 1,574 | RMSE: 4,820 | MAPE: 2.75% | MdAPE: 1.76% | RMSLE: 0.0442
------------------------------
FINAL TEST RESULT: R2: 0.9459 | MAE: 2,779 | RMSE: 7,125 | MAPE: 5.23% | MdAPE: 3.35% | RMSLE: 0.0745


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1)
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_부산광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (LightGBM)
print(f"{'='*30}\nSTART: LIGHTGBM (SAMPLE-WISE SPLIT)\n{'='*30}")

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,           # XGBoost와 동일하게 0.05
    num_leaves=31,                # LGBM의 핵심 파라미터 (max_depth와 대응)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    importance_type='gain',
    verbosity=-1                  # 불필요한 경고 메시지 출력 방지
)

# 6. 학습 (Early Stopping 설정 방식이 다름)
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20), # 여기서 조기종료 설정
        lgb.log_evaluation(period=0)            # verbose=False와 동일 효과
    ]
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LIGHTGBM (SAMPLE-WISE SPLIT)
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[66]	valid_0's rmse: 0.259948	valid_0's l2: 0.0675728
Validation 결과 | R2: 0.9790 | MAE: 1,548 | RMSE: 4,270 | MAPE: 3.34% | MdAPE: 2.04% | RMSLE: 0.0498
------------------------------
FINAL TEST RESULT: R2: 0.9499 | MAE: 2,547 | RMSE: 6,805 | MAPE: 5.18% | MdAPE: 3.37% | RMSLE: 0.0744


### 대구

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1)
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_대구광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (LightGBM)
print(f"{'='*30}\nSTART: LIGHTGBM (SAMPLE-WISE SPLIT)\n{'='*30}")

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,           # XGBoost와 동일하게 0.05
    num_leaves=31,                # LGBM의 핵심 파라미터 (max_depth와 대응)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    importance_type='gain',
    verbosity=-1                  # 불필요한 경고 메시지 출력 방지
)

# 6. 학습 (Early Stopping 설정 방식이 다름)
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20), # 여기서 조기종료 설정
        lgb.log_evaluation(period=0)            # verbose=False와 동일 효과
    ]
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LIGHTGBM (SAMPLE-WISE SPLIT)
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[179]	valid_0's rmse: 0.156571	valid_0's l2: 0.0245144
Validation 결과 | R2: 0.9828 | MAE: 975 | RMSE: 1,769 | MAPE: 3.90% | MdAPE: 2.82% | RMSLE: 0.0574
------------------------------
FINAL TEST RESULT: R2: 0.9421 | MAE: 2,011 | RMSE: 3,343 | MAPE: 7.62% | MdAPE: 5.36% | RMSLE: 0.1028


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1)
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_대구광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (LightGBM)
print(f"{'='*30}\nSTART: LIGHTGBM (SAMPLE-WISE SPLIT)\n{'='*30}")

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,           # XGBoost와 동일하게 0.05
    num_leaves=31,                # LGBM의 핵심 파라미터 (max_depth와 대응)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    importance_type='gain',
    verbosity=-1                  # 불필요한 경고 메시지 출력 방지
)

# 6. 학습 (Early Stopping 설정 방식이 다름)
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20), # 여기서 조기종료 설정
        lgb.log_evaluation(period=0)            # verbose=False와 동일 효과
    ]
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LIGHTGBM (SAMPLE-WISE SPLIT)
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[150]	valid_0's rmse: 0.143744	valid_0's l2: 0.0206624
Validation 결과 | R2: 0.9880 | MAE: 799 | RMSE: 1,768 | MAPE: 2.52% | MdAPE: 2.04% | RMSLE: 0.0339
------------------------------
FINAL TEST RESULT: R2: 0.9673 | MAE: 1,648 | RMSE: 2,776 | MAPE: 6.15% | MdAPE: 4.85% | RMSLE: 0.0753


In [ ]:
import os

import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1)
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_대구광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (LightGBM)
print(f"{'='*30}\nSTART: LIGHTGBM (SAMPLE-WISE SPLIT)\n{'='*30}")

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,           # XGBoost와 동일하게 0.05
    num_leaves=31,                # LGBM의 핵심 파라미터 (max_depth와 대응)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    importance_type='gain',
    verbosity=-1                  # 불필요한 경고 메시지 출력 방지
)

# 6. 학습 (Early Stopping 설정 방식이 다름)
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20), # 여기서 조기종료 설정
        lgb.log_evaluation(period=0)            # verbose=False와 동일 효과
    ]
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LIGHTGBM (SAMPLE-WISE SPLIT)
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[105]	valid_0's rmse: 0.127891	valid_0's l2: 0.016356
Validation 결과 | R2: 0.9902 | MAE: 700 | RMSE: 1,572 | MAPE: 2.29% | MdAPE: 1.71% | RMSLE: 0.0334
------------------------------
FINAL TEST RESULT: R2: 0.9738 | MAE: 1,307 | RMSE: 2,463 | MAPE: 4.83% | MdAPE: 3.42% | RMSLE: 0.0645


### 대전

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1)
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_대전광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (LightGBM)
print(f"{'='*30}\nSTART: LIGHTGBM (SAMPLE-WISE SPLIT)\n{'='*30}")

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,           # XGBoost와 동일하게 0.05
    num_leaves=31,                # LGBM의 핵심 파라미터 (max_depth와 대응)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    importance_type='gain',
    verbosity=-1                  # 불필요한 경고 메시지 출력 방지
)

# 6. 학습 (Early Stopping 설정 방식이 다름)
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20), # 여기서 조기종료 설정
        lgb.log_evaluation(period=0)            # verbose=False와 동일 효과
    ]
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LIGHTGBM (SAMPLE-WISE SPLIT)
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[199]	valid_0's rmse: 0.30467	valid_0's l2: 0.0928239
Validation 결과 | R2: 0.9657 | MAE: 1,636 | RMSE: 3,240 | MAPE: 5.33% | MdAPE: 4.21% | RMSLE: 0.0735
------------------------------
FINAL TEST RESULT: R2: 0.8969 | MAE: 3,459 | RMSE: 6,422 | MAPE: 9.86% | MdAPE: 7.69% | RMSLE: 0.1308


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1)
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_대전광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (LightGBM)
print(f"{'='*30}\nSTART: LIGHTGBM (SAMPLE-WISE SPLIT)\n{'='*30}")

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,           # XGBoost와 동일하게 0.05
    num_leaves=31,                # LGBM의 핵심 파라미터 (max_depth와 대응)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    importance_type='gain',
    verbosity=-1                  # 불필요한 경고 메시지 출력 방지
)

# 6. 학습 (Early Stopping 설정 방식이 다름)
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20), # 여기서 조기종료 설정
        lgb.log_evaluation(period=0)            # verbose=False와 동일 효과
    ]
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LIGHTGBM (SAMPLE-WISE SPLIT)
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[142]	valid_0's rmse: 0.184823	valid_0's l2: 0.0341596
Validation 결과 | R2: 0.9884 | MAE: 1,045 | RMSE: 2,464 | MAPE: 2.33% | MdAPE: 1.72% | RMSLE: 0.0329
------------------------------
FINAL TEST RESULT: R2: 0.9640 | MAE: 2,107 | RMSE: 4,145 | MAPE: 5.30% | MdAPE: 3.50% | RMSLE: 0.0717


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1)
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_대전광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (LightGBM)
print(f"{'='*30}\nSTART: LIGHTGBM (SAMPLE-WISE SPLIT)\n{'='*30}")

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,           # XGBoost와 동일하게 0.05
    num_leaves=31,                # LGBM의 핵심 파라미터 (max_depth와 대응)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    importance_type='gain',
    verbosity=-1                  # 불필요한 경고 메시지 출력 방지
)

# 6. 학습 (Early Stopping 설정 방식이 다름)
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20), # 여기서 조기종료 설정
        lgb.log_evaluation(period=0)            # verbose=False와 동일 효과
    ]
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LIGHTGBM (SAMPLE-WISE SPLIT)
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[67]	valid_0's rmse: 0.178312	valid_0's l2: 0.0317952
Validation 결과 | R2: 0.9887 | MAE: 1,026 | RMSE: 2,376 | MAPE: 2.73% | MdAPE: 2.06% | RMSLE: 0.0369
------------------------------
FINAL TEST RESULT: R2: 0.9683 | MAE: 1,811 | RMSE: 3,847 | MAPE: 4.88% | MdAPE: 3.23% | RMSLE: 0.0676


### 광주

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1)
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_광주광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (LightGBM)
print(f"{'='*30}\nSTART: LIGHTGBM (SAMPLE-WISE SPLIT)\n{'='*30}")

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,           # XGBoost와 동일하게 0.05
    num_leaves=31,                # LGBM의 핵심 파라미터 (max_depth와 대응)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    importance_type='gain',
    verbosity=-1                  # 불필요한 경고 메시지 출력 방지
)

# 6. 학습 (Early Stopping 설정 방식이 다름)
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20), # 여기서 조기종료 설정
        lgb.log_evaluation(period=0)            # verbose=False와 동일 효과
    ]
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LIGHTGBM (SAMPLE-WISE SPLIT)
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[137]	valid_0's rmse: 0.150755	valid_0's l2: 0.0227272
Validation 결과 | R2: 0.9862 | MAE: 896 | RMSE: 1,378 | MAPE: 4.66% | MdAPE: 3.50% | RMSLE: 0.0631
------------------------------
FINAL TEST RESULT: R2: 0.9305 | MAE: 2,129 | RMSE: 3,694 | MAPE: 8.36% | MdAPE: 6.04% | RMSLE: 0.1204


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1) # XGBoost 예측값 대응
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_광주광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (LightGBM)
print(f"{'='*30}\nSTART: LIGHTGBM (SAMPLE-WISE SPLIT)\n{'='*30}")

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,           # XGBoost와 동일하게 0.05
    num_leaves=31,                # LGBM의 핵심 파라미터 (max_depth와 대응)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    importance_type='gain',
    verbosity=-1                  # 불필요한 경고 메시지 출력 방지
)

# 6. 학습 (Early Stopping 설정 방식이 다름)
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20), # 여기서 조기종료 설정
        lgb.log_evaluation(period=0)            # verbose=False와 동일 효과
    ]
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LIGHTGBM (SAMPLE-WISE SPLIT)
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[205]	valid_0's rmse: 0.115232	valid_0's l2: 0.0132784
Validation 결과 | R2: 0.9939 | MAE: 592 | RMSE: 1,202 | MAPE: 2.37% | MdAPE: 1.95% | RMSLE: 0.0316
------------------------------
FINAL TEST RESULT: R2: 0.9737 | MAE: 1,470 | RMSE: 2,591 | MAPE: 5.40% | MdAPE: 3.79% | RMSLE: 0.0732


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    scaled_y = scaled_y.reshape(-1, 1)
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_광주광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
scaler = StandardScaler()
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class XGBoostDataset:
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        x_scaled = scaler.transform(x_raw.reshape(-1, F)).reshape(N, -1)
        region_idx = data_df['region_idx'].values.reshape(-1, 1)
        self.X = np.hstack([x_scaled, region_idx])
        
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = scaler.transform(dummy)[:, 0]

train_set = XGBoostDataset(train_df, scaler, len(FEATURES))
val_set = XGBoostDataset(val_df, scaler, len(FEATURES))
test_set = XGBoostDataset(test_df, scaler, len(FEATURES))

# 5. 모델 정의 (LightGBM)
print(f"{'='*30}\nSTART: LIGHTGBM (SAMPLE-WISE SPLIT)\n{'='*30}")

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,           # XGBoost와 동일하게 0.05
    num_leaves=31,                # LGBM의 핵심 파라미터 (max_depth와 대응)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    importance_type='gain',
    verbosity=-1                  # 불필요한 경고 메시지 출력 방지
)

# 6. 학습 (Early Stopping 설정 방식이 다름)
model.fit(
    train_set.X, train_set.y,
    eval_set=[(val_set.X, val_set.y)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=20), # 여기서 조기종료 설정
        lgb.log_evaluation(period=0)            # verbose=False와 동일 효과
    ]
)

# Validation 결과 확인용
v_p_scaled = model.predict(val_set.X)
v_p = get_inverse_price(v_p_scaled, scaler, len(FEATURES))
v_a = get_inverse_price(val_set.y, scaler, len(FEATURES))
print(f"Validation 결과 | {calculate_metrics(v_a, v_p)}")

# 7. 최종 결과
all_p = model.predict(test_set.X)
all_a = test_set.y

y_p = get_inverse_price(all_p, scaler, len(FEATURES))
y_a = get_inverse_price(all_a, scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LIGHTGBM (SAMPLE-WISE SPLIT)
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[137]	valid_0's rmse: 0.107936	valid_0's l2: 0.0116501
Validation 결과 | R2: 0.9944 | MAE: 588 | RMSE: 1,125 | MAPE: 2.38% | MdAPE: 1.89% | RMSLE: 0.0320
------------------------------
FINAL TEST RESULT: R2: 0.9753 | MAE: 1,395 | RMSE: 2,490 | MAPE: 5.23% | MdAPE: 3.70% | RMSLE: 0.0713
